In [ ]:
import zipfile
import pandas as pd
import numpy as np

# Unzip
with zipfile.ZipFile('ammonia.zip', 'r') as zip_ref:
    zip_ref.extractall('pond_data')

# Load one good pond file (Pond 2 is large and clean enough to start)
df = pd.read_csv('pond_data/IoTPond2.csv')
print(df.shape)
print(df.head())
print(df.columns.tolist())

(172249, 11)
                created_at  entry_id  Temperature (C)  Turbidity (NTU)  \
0  2021-06-19 00:00:05 CET      1889          24.8750              100   
1  2021-06-19 00:01:02 CET      1890          24.9375              100   
2  2021-06-19 00:01:22 CET      1891          24.8750              100   
3  2021-06-19 00:01:44 CET      1892          24.9375              100   
4  2021-06-19 00:02:07 CET      1893          24.9375              100   

   Dissolved Oxygen(g/ml)       PH  Ammonia(g/ml)  Nitrate(g/ml)  Population  \
0                   4.505  8.43365           0.38            193          50   
1                   6.601  8.43818           0.38            194          50   
2                  15.797  8.42457           0.38            192          50   
3                   5.046  8.43365           0.38            193          50   
4                  38.407  8.40641           0.38            192          50   

   Fish_Length (cm)  Fish_Weight (g)  
0              6.96   

In [ ]:
import numpy as np

# Rename columns for easier use
df = df.rename(columns={
    'Temperature (C)': 'temp',
    'Turbidity (NTU)': 'turbidity',
    'Dissolved Oxygen(g/ml)': 'do',
    'PH': 'ph',
    'Ammonia(g/ml)': 'ammonia',
    'Nitrate(g/ml)': 'nitrate'
})

# Convert time and sort
df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
df = df.sort_values('created_at').reset_index(drop=True)

# Remove bad values
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['temp', 'turbidity', 'do', 'ph', 'ammonia', 'nitrate'])

print("Cleaned shape:", df.shape)
print("\nAmmonia statistics:")
print(df['ammonia'].describe())

Cleaned shape: (172159, 11)

Ammonia statistics:
count    1.721590e+05
mean     3.112110e+08
std      1.257809e+10
min      6.590000e-03
25%      5.693500e-01
50%      8.470560e+00
75%      8.070516e+01
max      9.965130e+11
Name: ammonia, dtype: float64


/tmp/ipykernel_2851/4079311018.py:14: FutureWarning: Parsed string "2021-06-19 00:00:05 CET" included an un-recognized timezone "CET". Dropping unrecognized timezones is deprecated; in a future version this will raise. Instead pass the string without the timezone, then use .tz_localize to convert to a recognized timezone.
  df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')


In [ ]:
# Keep only realistic ammonia values (remove the extreme garbage)
df = df[df['ammonia'] < 100]   # can change 100 later if needed
df = df.reset_index(drop=True)

print("After removing extreme ammonia:")
print(df['ammonia'].describe())
print("\nRows left:", len(df))

# Look at higher percentiles to choose a spike threshold
print("\nPercentiles:")
print(df['ammonia'].quantile([0.75, 0.90, 0.95, 0.99]))

After removing extreme ammonia:
count    131682.000000
mean         12.525582
std          19.887489
min           0.006590
25%           0.380000
50%           3.334870
75%          14.921910
max          99.923010
Name: ammonia, dtype: float64

Rows left: 131682

Percentiles:
0.75    14.92191
0.90    40.33092
0.95    59.19246
0.99    88.89679
Name: ammonia, dtype: float64


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
import numpy as np

# Features (do not use ammonia itself as input)
features = ['temp', 'turbidity', 'do', 'ph', 'nitrate']
df['spike'] = (df['ammonia'] > 10).astype(int)

print("Spike count:", df['spike'].sum())
print("Spike percentage:", df['spike'].mean()*100)

# Scale features
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[features])

# Create sequences (use last 30 readings to predict next spike)
SEQ_LEN = 30
X, y = [], []

for i in range(len(scaled) - SEQ_LEN):
    X.append(scaled[i:i+SEQ_LEN])
    y.append(df['spike'].iloc[i+SEQ_LEN])

X = np.array(X)
y = np.array(y)

print("X shape:", X.shape)
print("y shape:", y.shape)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# GRU model
model = Sequential([
    GRU(64, input_shape=(SEQ_LEN, len(features))),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])
model.summary()

# Train
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

Spike count: 41550
Spike percentage: 31.553287465257213
X shape: (131652, 30, 5)
y shape: (131652,)


/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        13,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,745 (61.50 KB)

 Trainable params: 15,745 (61.50 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 22s 13ms/step - AUC: 0.8357 - accuracy: 0.7619 - loss: 0.4496 - val_AUC: 0.8923 - val_accuracy: 0.8196 - val_loss: 0.3810
Epoch 2/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - AUC: 0.8942 - accuracy: 0.8204 - loss: 0.3738 - val_AUC: 0.9007 - val_accuracy: 0.8231 - val_loss: 0.3650
Epoch 3/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - AUC: 0.9120 - accuracy: 0.8342 - loss: 0.3407 - val_AUC: 0.9246 - val_accuracy: 0.8407 - val_loss: 0.3146
Epoch 4/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 20s 13ms/step - AUC: 0.9178 - accuracy: 0.8389 - loss: 0.3283 - val_AUC: 0.9112 - val_accuracy: 0.8305 - val_loss: 0.3599
Epoch 5/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 21s 14ms/step - AUC: 0.9224 - accuracy: 0.8426 - loss: 0.3192 - val_AUC: 0.9213 - val_accuracy: 0.8406 - val_loss: 0.3269
Epoch 6/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - AUC: 0.9263 - accuracy: 0.8471 - loss: 0.3106 - val_AUC: 0.9297 - val_accuracy: 0.8465 - val_loss: 0.3071
Epoch 7/15
1482/

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred_prob = model.predict(X_test).flatten()
y_pred = (y_pred_prob > 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_prob))

823/823 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
              precision    recall  f1-score   support

           0       0.91      0.90      0.90     18021
           1       0.78      0.82      0.80      8310

    accuracy                           0.87     26331
   macro avg       0.85      0.86      0.85     26331
weighted avg       0.87      0.87      0.87     26331

ROC-AUC: 0.9452272288827896


In [ ]:
from tensorflow.keras.layers import LSTM

lstm_model = Sequential([
    LSTM(64, input_shape=(SEQ_LEN, len(features))),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy', 'AUC'])

lstm_history = lstm_model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

# Evaluate LSTM
lstm_pred_prob = lstm_model.predict(X_test).flatten()
lstm_pred = (lstm_pred_prob > 0.5).astype(int)

print("=== LSTM ===")
print(classification_report(y_test, lstm_pred))
print("ROC-AUC:", roc_auc_score(y_test, lstm_pred_prob))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 19s 12ms/step - AUC: 0.8123 - accuracy: 0.7500 - loss: 0.4738 - val_AUC: 0.8007 - val_accuracy: 0.7316 - val_loss: 0.4716
Epoch 2/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - AUC: 0.8533 - accuracy: 0.7780 - loss: 0.4258 - val_AUC: 0.8716 - val_accuracy: 0.7893 - val_loss: 0.4042
Epoch 3/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 17s 12ms/step - AUC: 0.8996 - accuracy: 0.8225 - loss: 0.3616 - val_AUC: 0.9131 - val_accuracy: 0.8326 - val_loss: 0.3499
Epoch 4/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 16s 11ms/step - AUC: 0.9140 - accuracy: 0.8385 - loss: 0.3371 - val_AUC: 0.9070 - val_accuracy: 0.8277 - val_loss: 0.3521
Epoch 5/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step - AUC: 0.9185 - accuracy: 0.8419 - loss: 0.3279 - val_AUC: 0.9082 - val_accuracy: 0.8321 - val_loss: 0.3520
Epoch 6/15
1482/1482 ━━━━━━━━━━━━━━━━━━━━ 21s 12ms/step - AUC: 0.9227 - accuracy: 0.8454 - loss: 0.3207 - val_AUC: 0.9170 - val_accuracy: 0.8346 - val_loss: 0.3285
Epoch 7/15
1482/

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Flatten the sequences for Random Forest
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_flat, y_train)

rf_pred = rf.predict(X_test_flat)
rf_pred_prob = rf.predict_proba(X_test_flat)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, rf_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf_pred_prob))

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     18021
           1       0.96      0.96      0.96      8310

    accuracy                           0.97     26331
   macro avg       0.97      0.97      0.97     26331
weighted avg       0.97      0.97      0.97     26331

ROC-AUC: 0.9966167362839355


In [ ]:
# Save GRU
model.save('ammonia_spike_gru.keras')

# Save LSTM
lstm_model.save('ammonia_spike_lstm.keras')

print("GRU and LSTM saved")

GRU and LSTM saved


In [ ]:
import joblib

joblib.dump(rf, 'ammonia_spike_rf_model.pkl')
print("Random Forest saved as ammonia_spike_rf_model.pkl")

Random Forest saved as ammonia_spike_rf_model.pkl


In [ ]:
import joblib
from tensorflow.keras.models import load_model
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score
import pandas as pd

# Load models
gru_model = load_model('ammonia_spike_gru.keras')
lstm_model = load_model('ammonia_spike_lstm.keras')
rf_model = joblib.load('ammonia_spike_rf_model.pkl')

# Predictions
gru_prob = gru_model.predict(X_test).flatten()
lstm_prob = lstm_model.predict(X_test).flatten()

X_test_flat = X_test.reshape(X_test.shape[0], -1)
rf_prob = rf_model.predict_proba(X_test_flat)[:, 1]

# Helper function
def get_metrics(y_true, y_prob, name):
    y_pred = (y_prob > 0.5).astype(int)
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_true, y_pred), 4),
        'Precision (spike)': round(precision_score(y_true, y_pred), 4),
        'Recall (spike)': round(recall_score(y_true, y_pred), 4),
        'F1-score (spike)': round(f1_score(y_true, y_pred), 4),
        'ROC-AUC': round(roc_auc_score(y_true, y_prob), 4)
    }

# Compare
results = [
    get_metrics(y_test, gru_prob, 'GRU'),
    get_metrics(y_test, lstm_prob, 'LSTM'),
    get_metrics(y_test, rf_prob, 'Random Forest')
]

comparison = pd.DataFrame(results)
print(comparison.to_string(index=False))

823/823 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
823/823 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step
        Model  Accuracy  Precision (spike)  Recall (spike)  F1-score (spike)  ROC-AUC
          GRU    0.8710             0.7846          0.8152            0.7996   0.9452
         LSTM    0.8654             0.7785          0.8017            0.7899   0.9455
Random Forest    0.9742             0.9578          0.9605            0.9591   0.9966


In [ ]:
# Example: take the first test sample
sample = X_test[0:1]          # shape (1, 30, 5)

# GRU prediction
gru_pred = gru_model.predict(sample)[0][0]
print("GRU probability of spike:", round(gru_pred, 3))
print("GRU prediction:", "Spike" if gru_pred > 0.5 else "Normal")

# LSTM prediction
lstm_pred = lstm_model.predict(sample)[0][0]
print("LSTM probability of spike:", round(lstm_pred, 3))
print("LSTM prediction:", "Spike" if lstm_pred > 0.5 else "Normal")

# Random Forest prediction
sample_flat = sample.reshape(1, -1)
rf_pred = rf_model.predict_proba(sample_flat)[0][1]
print("RF probability of spike:", round(rf_pred, 3))
print("RF prediction:", "Spike" if rf_pred > 0.5 else "Normal")

print("\nActual label:", "Spike" if y_test[0] == 1 else "Normal")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
GRU probability of spike: 0.826
GRU prediction: Spike
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
LSTM probability of spike: 0.749
LSTM prediction: Spike
RF probability of spike: 0.85
RF prediction: Spike

Actual label: Spike


In [ ]:
import os
print(os.listdir())

['.config', 'ammonia_spike_gru.keras', 'ammonia_spike_rf_model.pkl', 'ammonia_spike_lstm.keras', 'ammonia.zip', 'pond_data', 'sample_data']


In [ ]:
print(type(rf))
print(type(gru_model))
print(type(lstm_model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
<class 'keras.src.models.sequential.Sequential'>
<class 'keras.src.models.sequential.Sequential'>
